# 50. Conversation Memory

<a href="https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/06-iterative/50_conversation_memory.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Category:** 06 - Iterative & Conversational  **Technique:** #50 - Conversation Memory

---

## 📋 Description

**Conversation Memory** is a technique for maintaining and recalling information across multiple conversation turns. It enables AI systems to remember user preferences, facts, and context from earlier in the conversation, creating more personalized and coherent interactions.

### When to Use:
- Long-running conversations
- Personal assistant applications
- Customer support with history
- Learning/tutoring systems
- Any multi-session interaction

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────┐
│                    CONVERSATION MEMORY                   │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐ │
│  │  Facts      │    │ Preferences │    │  Context    │ │
│  │  - Name     │    │  - Style    │    │  - Topic    │ │
│  │  - Location │    │  - Format   │    │  - Goal     │ │
│  │  - History  │    │  - Tone     │    │  - State    │ │
│  └─────────────┘    └─────────────┘    └─────────────┘ │
│                                                          │
│  Memory Types:                                          │
│  - Short-term: Recent conversation turns                │
│  - Long-term: User profile, persistent preferences      │
│  - Working: Current task context                        │
│                                                          │
└─────────────────────────────────────────────────────────┘
```

### Memory Categories:
1. **Explicit Memory** - Directly stated facts
2. **Implicit Memory** - Inferred preferences
3. **Working Memory** - Current task context
4. **Episodic Memory** - Past conversation events

## ⚙️ Setup

Install required packages and configure API access.

In [ ]:
# Install required packages
!pip install openai -q

import os
from getpass import getpass
from openai import OpenAI
from datetime import datetime

# Secure API key input
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize client
client = OpenAI()

print("✅ Setup complete!")

## 🎯 Basic Example

Simple conversation memory implementation.

In [ ]:
class ConversationMemory:
    """
    Manages conversation memory with fact extraction.
    """
    
    def __init__(self, model="gpt-4o"):
        self.model = model
        self.messages = []
        self.facts = {}
        self.preferences = {}
    
    def send_message(self, user_message, extract_memory=True):
        """Send message and maintain memory."""
        
        # Include memory context
        memory_context = self._build_memory_context()
        
        messages = []
        if memory_context:
            messages.append({
                "role": "system", 
                "content": f"Remember these facts about the user: {memory_context}"
            })
        
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})
        
        response = client.chat.completions.create(
            model=self.model,
            messages=messages
        )
        
        assistant_message = response.choices[0].message.content
        
        # Update conversation history
        self.messages.extend([
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ])
        
        # Extract memory if enabled
        if extract_memory:
            self._extract_memory(user_message, assistant_message)
        
        return assistant_message
    
    def _build_memory_context(self):
        """Build context string from memory."""
        context_parts = []
        
        if self.facts:
            facts_str = ", ".join([f"{k}={v}" for k, v in self.facts.items()])
            context_parts.append(f"Facts: {facts_str}")
        
        if self.preferences:
            prefs_str = ", ".join([f"{k}={v}" for k, v in self.preferences.items()])
            context_parts.append(f"Preferences: {prefs_str}")
        
        return "; ".join(context_parts)
    
    def _extract_memory(self, user_msg, assistant_msg):
        """Extract facts and preferences from conversation."""
        # Simple extraction - in production, use more sophisticated NLP
        if "my name is " in user_msg.lower():
            name = user_msg.lower().split("my name is ")[1].split()[0].strip(".!")
            self.facts["name"] = name.capitalize()
        elif "i'm " in user_msg.lower() and "name" not in self.facts:
            parts = user_msg.lower().split("i'm ")[1].split()
            if parts:
                self.facts["name"] = parts[0].capitalize().strip(".!")
    
    def get_memory(self):
        """Return current memory state."""
        return {
            "facts": self.facts,
            "preferences": self.preferences,
            "message_count": len(self.messages) // 2
        }

# Example usage
print("=" * 60)
print("CONVERSATION MEMORY EXAMPLE")
print("=" * 60 + "\n")

memory_chat = ConversationMemory()

# Turn 1: Share information
response1 = memory_chat.send_message("Hi, I'm Sarah and I work as a data scientist.")
print(f"User: Hi, I'm Sarah and I work as a data scientist.")
print(f"AI: {response1}\n")

# Turn 2: Reference previous info
response2 = memory_chat.send_message("What do you think about my career path?")
print(f"User: What do you think about my career path?")
print(f"AI: {response2[:200]}...\n")

# Turn 3: AI remembers details
response3 = memory_chat.send_message("Can you recommend some skills I should learn?")
print(f"User: Can you recommend some skills I should learn?")
print(f"AI: {response3[:200]}...\n")

# Show memory state
print("Memory State:")
print(memory_chat.get_memory())

## 💼 Real-World Example

Personal assistant with persistent memory across sessions.

In [ ]:
# Simulate persistent memory across a conversation
print("=" * 60)
print("PERSONAL ASSISTANT WITH MEMORY")
print("=" * 60 + "\n")

# Simulate a multi-turn conversation with memory
conversation = [
    "Hi, I'm Alex and I live in Seattle.",
    "I need help planning a trip to Portland next weekend.",
    "What should I pack for the weather?",
    "Also, I prefer casual dining over fancy restaurants.",
    "Can you recommend some places to eat?"
]

assistant = ConversationMemory()

for i, message in enumerate(conversation, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {message}")
    
    response = assistant.send_message(message)
    print(f"AI: {response[:200]}...")
    
    # Show accumulated memory
    memory = assistant.get_memory()
    print(f"\n[Memory: {memory}]")

print("\n" + "=" * 60)
print("FINAL MEMORY STATE:")
print("=" * 60)
print(assistant.get_memory())

## ⚠️ Failure Case

Common memory failures and solutions.

In [ ]:
# ❌ BAD: Memory overflow
print("❌ BAD PRACTICE - Memory Overflow:\n")

print("""
Scenario: Conversation with 100+ turns

BAD APPROACH:
- Include ALL previous messages in every API call
- Token count grows with each turn
- Eventually exceeds context window
- API calls become slow and expensive

PROBLEM: Unbounded memory growth
""")

# ✅ GOOD: Memory management
print("\n✅ GOOD PRACTICE - Memory Management:\n")

print("""
GOOD APPROACH:
- Keep only recent N messages (e.g., last 10)
- Summarize older conversations
- Extract key facts to separate memory store
- Compress redundant information

BENEFIT: Bounded memory, faster responses
""")

print("\n" + "=" * 60)
print("MEMORY MANAGEMENT STRATEGIES:")
print("=" * 60)
print("""

1. SLIDING WINDOW
   - Keep last N message pairs
   - Discard older messages
   - Simple but loses old context

2. SUMMARIZATION
   - Periodically summarize old conversation
   - Replace detailed messages with summary
   - Preserves key information

3. FACT EXTRACTION
   - Extract important facts to key-value store
   - Reference facts in system prompt
   - Most efficient for long-term memory

4. HYBRID APPROACH
   - Recent messages: full detail
   - Medium-term: summary
   - Long-term: fact store
   - Best of all worlds

""")

# Demonstrate hybrid memory management
print("\n" + "=" * 60)
print("DEMONSTRATION - MEMORY MANAGEMENT:")
print("=" * 60 + "\n")

class ManagedMemory:
    def __init__(self, max_recent=6):
        self.max_recent = max_recent
        self.recent_messages = []
        self.summary = ""
        self.key_facts = {}
    
    def add_message(self, role, content):
        self.recent_messages.append({"role": role, "content": content})
        
        # Trim if exceeds limit
        if len(self.recent_messages) > self.max_recent:
            # Move oldest to summary (simplified)
            old = self.recent_messages.pop(0)
            self.summary += f"{old['role']}: {old['content'][:50]}... "
    
    def get_context(self):
        context = []
        if self.summary:
            context.append({"role": "system", "content": f"Previous conversation: {self.summary}"})
        if self.key_facts:
            facts = ", ".join([f"{k}={v}" for k, v in self.key_facts.items()])
            context.append({"role": "system", "content": f"Facts: {facts}"})
        context.extend(self.recent_messages)
        return context

# Demo
mm = ManagedMemory(max_recent=4)

messages = [
    ("user", "Hi, I'm John"),
    ("assistant", "Hello John!"),
    ("user", "I need help with Python"),
    ("assistant", "I'd be happy to help!"),
    ("user", "How do I use dictionaries?"),
    ("assistant", "Dictionaries store key-value pairs..."),
    ("user", "Can you show an example?"),
    ("assistant", "Sure: my_dict = {'a': 1}"),
]

for role, content in messages:
    mm.add_message(role, content)

print("After 8 messages with max_recent=4:")
print(f"Summary: {mm.summary[:100]}...")
print(f"Recent messages count: {len(mm.recent_messages)}")
print(f"Full context message count: {len(mm.get_context())}")

## 📊 Benchmark

| Memory Type | Context Retention | Token Cost | Best For |
|-------------|------------------|------------|----------|
| Full History | 100% | Very High | Short conversations |
| Sliding Window | 40-60% | Medium | Balanced use |
| Summarization | 70-80% | Low-Medium | Medium conversations |
| Fact Extraction | 60-70% | Low | Long-term memory |
| Hybrid | 85-90% | Medium | Most applications |

**Key Findings:**
- Hybrid approach offers best balance
- Fact extraction crucial for personalization
- Sliding window sufficient for most use cases

## 🎮 Interactive Playground

Experiment with conversation memory.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 🎮 INTERACTIVE PLAYGROUND - Conversation Memory
# ═══════════════════════════════════════════════════════════

# Create your own conversation
YOUR_CONVERSATION = [
    "Hi, I'm Maria from Brazil.",
    "I'm learning to code.",
    "What's the best language for beginners?",
    "I want to build web applications."
]

print("=" * 60)
print("CONVERSATION MEMORY PLAYGROUND")
print("=" * 60 + "\n")

playground_memory = ConversationMemory()

for i, message in enumerate(YOUR_CONVERSATION, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {message}")
    
    response = playground_memory.send_message(message)
    print(f"AI: {response[:180]}...")

print("\n" + "=" * 60)
print("EXTRACTED MEMORY:")
print("=" * 60)
print(playground_memory.get_memory())

# Test memory recall
print("\n" + "=" * 60)
print("TESTING MEMORY RECALL:")
print("=" * 60)

recall_test = "Based on what you know about me, what should I focus on learning?"
print(f"\nUser: {recall_test}")
recall_response = playground_memory.send_message(recall_test)
print(f"AI: {recall_response}")

## 💡 Tips & Tricks

### Best Practices:

1. **Prioritize Facts** - Names, preferences, key details
2. **Summarize Regularly** - Compress old conversations
3. **Explicit Confirmation** - Confirm important facts
4. **Graceful Degradation** - Work without perfect memory

### Memory Implementation Checklist:

```
□ Store user facts explicitly
□ Track conversation topics
□ Remember user preferences
□ Manage context window
□ Summarize old conversations
□ Handle memory failures gracefully
□ Allow memory correction
□ Respect privacy (forget on request)
```

### Storage Options:

| Storage | Persistence | Speed | Use Case |
|---------|-------------|-------|----------|
| In-Memory | Session only | Fastest | Development |
| File/JSON | Permanent | Fast | Simple apps |
| Database | Permanent | Medium | Production |
| Vector DB | Permanent | Medium | Semantic search |
| Redis | Configurable | Very Fast | Caching |

## 📚 References

1. [LangChain Memory Module](https://python.langchain.com/docs/modules/memory/)
2. [OpenAI - Context Management](https://platform.openai.com/docs/guides/chat-completions/managing-conversation-context)
3. [Building Chatbots with Memory](https://huggingface.co/blog/building-chatbots-with-memory)
4. [Vector Databases for Memory](https://www.pinecone.io/learn/vector-database/)